# Работа с Excel

Материалы:
* Макрушин С.В. Лекция 7: Работа с Excel
* https://docs.xlwings.org/en/stable/quickstart.html
* https://nbviewer.jupyter.org/github/pybokeh/jupyter_notebooks/blob/master/xlwings/Excel_Formatting.ipynb#search_text


## Задачи для совместного разбора

1. На листе "Рецептура" файла `себестоимостьА_в1.xlsx` для области "Пшеничный хлеб" рассчитать себестоимость всех видов продукции.

2. Результаты расчетов 1.1 сохранить в отдельном столбце области "Пшеничный хлеб"

3. Приблизить форматирование столбца, добавленного в задаче 2 к оформлению всей области.

4. Выполнить 3 с помощью "протягиваемых" формул.

## Лабораторная работа 7.1

1. Загрузите данные из файлов `reviews_sample.csv` (__ЛР2__) и `recipes_sample.csv` (__ЛР5__) в виде `pd.DataFrame`. Обратите внимание на корректное считывание столбца(ов) с индексами. Оставьте в таблице с рецептами следующие столбцы: `id`, `name`, `minutes`, `submitted`, `description`, `n_ingredients`

In [9]:
import pandas as pd
import xlwings as xw
import csv
import matplotlib.pyplot as plt


recipes_df = pd.read_csv(r'/Users/iliaegorov/Projects/fa/datat/02_pandas/data/recipes_sample.csv')
reviews_df = pd.read_csv(r'/Users/iliaegorov/Projects/fa/datat/02_pandas/data/reviews_sample.csv')
recipes_df = recipes_df[['id', 'name', 'minutes', 'submitted', 'description', 'n_ingredients']]
recipes_df

,id,name,minutes,submitted,description,n_ingredients
0,44123,george s at the cove black bean soup,90,2002-10-25,an original recipe created by chef scott meska...,18.0
1,67664,healthy for them yogurt popsicles,10,2003-07-26,my children and their friends ask for my homem...,NaN
2,38798,i can t believe it s spinach,30,2002-08-29,"these were so go, it surprised even me.",8.0
3,35173,italian gut busters,45,2002-07-27,my sister-in-law made these for us at a family...,NaN
4,84797,love is in the air beef fondue sauces,25,2004-02-23,i think a fondue is a very romantic casual din...,NaN
...,...,...,...,...,...,...
29995,267661,zurie s holey rustic olive and cheddar bread,80,2007-11-25,this is based on a french recipe but i changed...,10.0
29996,386977,zwetschgenkuchen bavarian plum cake,240,2009-08-24,"this is a traditional fresh plum cake, thought...",11.0
29997,103312,zwiebelkuchen southwest german onion cake,75,2004-11-03,this is a traditional late summer early fall s...,NaN
29998,486161,zydeco soup,60,2012-08-29,this is a delicious soup that i originally fou...,NaN


2. Случайным образом выберите 5% строк из каждой таблицы и сохраните две таблицы на разные листы в один файл `recipes.xlsx`. Дайте листам названия "Рецепты" и "Отзывы", соответствующие содержанию таблиц. 

In [4]:
recipes_5pct = recipes_df.sample(frac=0.05, random_state=42)
reviews_5pct = reviews_df.sample(frac=0.05, random_state=42)

with pd.ExcelWriter('recipes.xlsx', engine='openpyxl') as writer:
    recipes_5pct.to_excel(writer, sheet_name='Рецепты', index=False)
    reviews_5pct.to_excel(writer, sheet_name='Отзывы', index=False)

3. Используя `xlwings`, добавьте на лист `Рецепты` столбец `seconds_assign`, показывающий время выполнения рецепта в секундах. Выполните задание при помощи присваивания массива значений диапазону ячеек.

In [ ]:
app = xw.App(visible=True)
wb = app.books.open('recipes.xlsx')
ws = wb.sheets['Рецепты']

tbl = ws.range('A1').expand('table')
n_rows = tbl.shape[0]
last_col = tbl.shape[1]

mins = ws.range((2, 3), (n_rows, 3)).value
secs = [[m * 60 if isinstance(m, (int, float)) else 0] for m in mins]

ws.cells(1, last_col + 1).value = 'seconds_assign'
ws.range((2, last_col + 1), (n_rows, last_col + 1)).value = secs

4. Используя `xlwings`, добавьте на лист `Рецепты` столбец `seconds_formula`, показывающий время выполнения рецепта в секундах. Выполните задание при помощи формул Excel.

In [ ]:
col_m = xw.utils.colnum_string(3)
col_sec_f = last_col + 2
ws.cells(1, col_sec_f).value = 'seconds_formula'
ws.range((2, col_sec_f), (n_rows, col_sec_f)).formula = f'={col_m}2*60'

5. Сделайте названия всех добавленных столбцов полужирными и выровняйте по центру ячейки.

In [ ]:
for cell in ws.range('A1').expand('right'):
    cell.font.bold = True
    cell.api.HorizontalAlignment = -4108

6. Раскрасьте ячейки столбца `minutes` в соответствии со следующим правилом: если рецепт выполняется быстрее 5 минут, то цвет - зеленый; от 5 до 10 минут - жёлтый; и больше 10 - красный.

In [ ]:
for i in range(2, n_rows + 1):
    val = ws.cells(i, 3).value
    if val < 5: ws.cells(i, 3).color = (0, 255, 0)
    elif val <= 10: ws.cells(i, 3).color = (255, 255, 0)
    else: ws.cells(i, 3).color = (255, 0, 0)

7. Добавьте на лист `Рецепты`  столбец `n_reviews`, содержащий кол-во отзывов для этого рецепта. Выполните задание при помощи формул Excel.

In [ ]:
col_nrev = last_col + 3
ws.cells(1, col_nrev).value = 'n_reviews'
ws.range((2, col_nrev), (n_rows, col_nrev)).formula = '=COUNTIF(Отзывы!C:C, A2)'

## Лабораторная работа 7.2

8. Напишите функцию `validate()`, которая проверяет соответствие всех строк из листа `Отзывы` следующим правилам:
    * Рейтинг - это число от 0 до 5 включительно
    * Соответствующий рецепт имеется на листе `Рецепты`
    
В случае несоответствия этим правилам, выделите строку красным цветом

In [ ]:
def validate(wb):
    ws_rev = wb.sheets['Отзывы']
    ws_rec = wb.sheets['Рецепты']
    valid_ids = set(ws_rec.range('A2:A' + str(ws_rec.range('A1').expand('table').last_cell.row)).value)
    n_rev = ws_rev.range('A1').expand('table').shape[0]
    h = ws_rev.range('A1').expand('right').value
    c_rid, c_rat = h.index('recipe_id') + 1, h.index('rating') + 1

    for i in range(2, n_rev + 1):
        r = ws_rev.cells(i, c_rat).value
        rid = ws_rev.cells(i, c_rid).value
        if not (isinstance(r, (int, float)) and 0 <= r <= 5) or rid not in valid_ids:
            ws_rev.range(f'A{i}:{xw.utils.colnum_string(len(h))}{i}').color = (255, 0, 0)

validate(wb)

9. В файле `recipes_model.csv` находится модель данных предметной области "рецепты". При помощи пакета `csv` считайте эти данные. При помощи пакета `xlwings` запишите данные на лист `Модель` книги `recipes_model.xlsx`, начиная с ячейки `A2`, не используя циклы. Сделайте скриншот текущего состояния листа и прикрепите в ячейку ноутбука. 

In [ ]:
wb2 = app.books.add()
ws2 = wb2.sheets[0]
ws2.name = 'Модель'

with open('recipes_model.csv', 'r', encoding='utf-8') as f:
    data = list(csv.reader(f))

ws2.range('A1').value = data[0]
ws2.range('A2').value = data[1:]

10. При помощи пакета `xlwings` добавьте в столбец J формулу для описания столбца на языке SQL. Формула должна реализовывать следующую логику:

    1\. в начале строки идут значения из столбцов В и C (значение столбца С приведено к верхнему регистру), разделенные пробелом
    
    2\. далее идут слова на основе столбца "Ключ"
        2.1 если в столбце "Ключ" указано значение "PK", то дальше через пробел идет ключевое слово "PRIMARY KEY"
        2.2 если в столбце "Ключ" указано значение "FK", то дальше через пробел идет ключевое слово "REFERENCES", затем значения столбцов H и I в формате "название_таблицы(название_столбца)"
        
    3\. если в столбце "Обязательно к заполнению" указано значение "Y" и в столбце "Ключ" указано не "PK", то дальше через пробел идет ключевое слово "NOT NULL".

Заполните этой формулой необходимое количество строк, используя "протягивание". Количество строк для протягивания определите на основе данных.

Сделайте скриншот текущего состояния листа и прикрепите в ячейку ноутбука.

In [ ]:
n_data = len(data) - 1
sql_f = '=B2 & " " & UPPER(C2) & IF(E2="PK", " PRIMARY KEY", IF(E2="FK", " REFERENCES " & H2 & "(" & I2 & ")", "")) & IF(AND(D2="Y", E2<>"PK"), " NOT NULL", "")'
ws2.range(f'J2:J{n_data + 1}').formula = sql_f

11. При помощи пакета `xlwings` измените стилизацию листа `Модель`.
* для заголовков добавьте заливку цвета `00ccff`
* примените автоподбор ширины столбца;
* сделайте шрифт заголовков полужирным;
* добавьте таблице автофильтр.

Сделайте скриншот текущего состояния листа и прикрепите в ячейку ноутбука.

In [ ]:
for cell in ws2.range('A1:J1'):
    cell.font.bold = True
    cell.color = (0, 204, 255)
ws2.autofit('c')
ws2.range('A1').expand('table').api.AutoFilter

12. Посчитайте количество атрибутов для каждой из сущностей. Создайте лист `Статистика` и запишите в него результат группировки, начиная с ячейки "А1". Визуализируйте полученный результат при помощи столбчатой диаграммы. Сохраните полученную визуализацию на лист `Статистика`, начиная с ячейки "E2".  Сделайте скриншот листа `Статистика` и прикрепите в ячейку ноутбука.

* Вы можете воспользоваться методами для визуализации, которые поставляются вместе с объектами `pandas` (см. https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot) 

In [ ]:
model_df = pd.DataFrame(data[1:], columns=data[0])
stats = model_df.iloc[:, 0].value_counts().reset_index()
stats.columns = ['Сущность', 'Кол-во атрибутов']

ws3 = wb2.sheets.add('Статистика')
ws3.range('A2').value = stats.values.tolist()
ws3.range('A1:B1').value = ['Сущность', 'Кол-во атрибутов']

fig, ax = plt.subplots(figsize=(6, 4))
stats.plot.bar(x='Сущность', y='Кол-во атрибутов', ax=ax, legend=False, color='#00ccff')
plt.ylabel('Кол-во атрибутов')
plt.tight_layout()
plt.savefig('stats.png', dpi=150)
ws3.pictures.add('stats.png', left=ws3.range('E2').left, top=ws3.range('E2').top, update=True)

wb2.save('recipes_model.xlsx')
wb.save('recipes.xlsx')
app.quit()